# 03 Evaluation and error analysis
The test period is evaluated once by src/train.py. This notebook reads the saved results and predictions.

Make the project folder importable.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

Import libraries and project modules.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src import config, data, evaluate

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

import json

from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve

Load test metrics, model metadata and saved test predictions.

In [ ]:
with open(config.REPORTS_DIR / "test_metrics.json", "r", encoding="utf-8") as handle:
    tm = json.load(handle)
with open(config.META_PATH, "r", encoding="utf-8") as handle:
    meta = json.load(handle)
pred = pd.read_csv(config.REPORTS_DIR / "test_predictions.csv", parse_dates=["appointment_day"])
print("final run id:", meta["mlflow_run_id"])
print("test period:", tm["test_period"], "| rows:", tm["test_rows"], "| days:", tm["test_days"])

Show headline test metrics at the operating point for the model and every baseline.

In [ ]:
rows = {"model": tm["model_at_flag_rate"], **{f"baseline: {k}": v for k, v in tm["baselines_at_flag_rate"].items()}}
headline = pd.DataFrame(rows).T[["flag_rate", "prevalence", "precision_at_flag", "recall_at_flag", "lift_at_flag", "pr_auc", "roc_auc"]].round(4)
display(headline)

Check the charter success and kill criteria against the test result.

In [ ]:
print("criteria:", tm["criteria"])
print("charter lift target:", config.MIN_LIFT, "| kill lift:", config.KILL_LIFT)
print("model at threshold:", {k: round(v, 4) if isinstance(v, float) else v for k, v in tm["model_at_threshold"].items()})

Translate the result into capacity and cost terms.

In [ ]:
precision = tm["model_at_flag_rate"]["precision_at_flag"]
print("calls per day at the flag rate:", round(tm["calls_per_day_at_flag_rate"], 1))
print("no-shows reached per 100 calls (model):", round(precision * 100, 1))
print("no-shows reached per 100 calls (rule):", round(tm["baselines_at_flag_rate"]["rule"]["precision_at_flag"] * 100, 1))
print("no-shows reached per 100 calls (random):", round(tm["model_at_flag_rate"]["prevalence"] * 100, 1))

Show the break-even reminder success rate for several assumed missed-slot costs.

In [ ]:
rows = []
for ratio in [2, 5, 10, 20]:
    needed = 1.0 / (precision * ratio)
    rows.append({"missed_slot_cost_in_calls": ratio, "break_even_recovery_rate": "above 100%" if needed > 1 else f"{needed:.1%}"})
display(pd.DataFrame(rows))

Plot the precision-recall curve on the test period with the operating point.

In [ ]:
p, r, _ = precision_recall_curve(pred["no_show"], pred["probability"])
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(r, p, label="model")
ax.axhline(tm["model_at_flag_rate"]["prevalence"], color="grey", linestyle="--", label="base rate")
ax.scatter([tm["model_at_threshold"]["recall_at_threshold"]], [tm["model_at_threshold"]["precision_at_threshold"]], color="red", zorder=3, label="operating point")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.legend()
plt.show()

Plot calibration of predicted probabilities on the test period.

In [ ]:
frac, mean_pred = calibration_curve(pred["no_show"], pred["probability"], n_bins=10, strategy="quantile")
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot(mean_pred, frac, marker="o")
ax.plot([0, max(mean_pred.max(), frac.max())], [0, max(mean_pred.max(), frac.max())], linestyle="--", color="grey")
ax.set_xlabel("mean predicted probability")
ax.set_ylabel("observed no-show rate")
plt.show()

Show the confusion matrix saved by training.

In [ ]:
from IPython.display import Image

Image(filename=str(config.REPORTS_DIR / "confusion_matrix.png"))

Show slice analysis for every meaningful group.

In [ ]:
slices = pd.read_csv(config.REPORTS_DIR / "slice_analysis.csv")
display(slices.round(3))

Plot recall and flag rate by slice to see who the model serves worse.

In [ ]:
big = slices[slices["n"] >= 100].copy()
big["label"] = big["slice"] + ": " + big["value"].astype(str)
fig, axes = plt.subplots(1, 2, figsize=(13, max(4, 0.28 * len(big))), sharey=True)
axes[0].barh(big["label"], big["recall"])
axes[0].set_title("Recall by slice")
axes[1].barh(big["label"], big["flag_rate"])
axes[1].set_title("Flag rate by slice")
fig.tight_layout()
plt.show()

Summarise the largest gaps across groups.

In [ ]:
display(pd.read_csv(config.REPORTS_DIR / "fairness_gaps.csv").round(3))

Profile the no-shows the model caught against the ones it missed.

In [ ]:
table, _ = data.build_model_table()
test = table[table["split"] == "test"].merge(pred[["appointment_id", "probability", "flag"]], on="appointment_id")
noshows = test[test["no_show"] == 1].copy()
noshows["outcome"] = np.where(noshows["flag"] == 1, "caught", "missed")
display(noshows.groupby("outcome")[["age", "lead_days", "prior_appts", "prior_noshow_rate", "scheduled_hour"]].mean().round(2))
print(noshows["outcome"].value_counts().to_dict())

List the neighbourhoods with the most missed no-shows.

In [ ]:
missed = noshows[noshows["outcome"] == "missed"]
summary = pd.DataFrame({
    "missed": missed["neighbourhood"].value_counts(),
    "all_noshows": noshows["neighbourhood"].value_counts(),
}).fillna(0).astype(int)
summary["miss_share"] = (summary["missed"] / summary["all_noshows"]).round(3)
display(summary.sort_values("missed", ascending=False).head(10))

Show permutation importance computed on the test period by training.

In [ ]:
display(pd.read_csv(config.REPORTS_DIR / "permutation_importance.csv").round(5))
Image(filename=str(config.REPORTS_DIR / "permutation_importance.png"))

Run the drift check: PSI per feature between the training period and the later test period.

In [ ]:
drift = pd.read_csv(config.REPORTS_DIR / "psi_train_vs_test.csv")
display(drift.round(4))
print("features above the 0.25 alarm level:", drift.loc[drift["psi"] > 0.25, "feature"].tolist())

## Findings
Write 5 to 8 sentences here after running: the recommendation, how the result compares with the rule, who is served worse, what the drift check shows, and what data would change the answer.